# Notebook #1: Loading Data

## 1.0 Setup
### 1.1 Pip Installs

In [16]:
# %pip install plotly

### 1.2 Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

## 2.0 Load the Data

The data are loaded from `ds08a_rows.parquet`, which was rebuilt directly from NASA's original
N-CMAPSS HDF5 file (`N-CMAPSS_DS08a-009.h5`) using `build_ds08a_from_h5.py`.

Compared with the earlier `ds08a.csv`, the rebuilt file:
* Keeps the same downsampling
    * 1 sample per 76 seconds for Flight Class 1 (`Fc == 1`)
    * 127s for FC2
    * 180s for FC3
    * and the same sensor and operating-condition values,
* Fills in the 10 degradation columns (`*_mod`), which had only 14 non-missing values in an earlier extracted CSV because a separately sampled table was attached by row position;
* Adds `split` (NASA's original `dev` / `test` label), `raw_idx` (row position in the original 1 Hz file), and `RUL` (remaining useful life, not used in this project);
* Names the engineered **Engine Pressure Ratio column** `EPR` (previously `eng_psia_ratio`); and
* Drops the generated `timestamp` column, which did not reflect real time.

In [3]:
df = pd.read_parquet("data/ds08a_rows.parquet")

print(df.shape)
df.head()

(67328, 50)


,split,raw_idx,alt,Mach,TRA,T2,T24,T30,T48,T50,...,HPT_eff_mod,HPT_flow_mod,LPT_eff_mod,LPT_flow_mod,RUL,unit,cycle,Fc,hs,EPR
0,dev,0,3003.0,0.392490,84.286507,523.610962,632.830627,1525.667236,1953.949585,1319.505371,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.144806
1,dev,76,3757.0,0.414981,84.286507,522.673950,631.582214,1521.578369,1947.109619,1312.881836,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.134743
2,dev,152,4676.0,0.417879,76.024849,519.527405,620.124512,1482.610229,1877.180664,1275.688599,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.103322
3,dev,228,5523.0,0.430164,77.079529,517.440247,618.549683,1480.296631,1875.717041,1271.735474,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.101272
4,dev,304,6591.0,0.427392,77.958427,513.253784,614.360046,1473.019409,1868.009644,1264.644653,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.105557


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 67328 entries, 0 to 67327
Data columns (total 50 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   split         67328 non-null  str    
 1   raw_idx       67328 non-null  int64  
 2   alt           67328 non-null  float32
 3   Mach          67328 non-null  float32
 4   TRA           67328 non-null  float32
 5   T2            67328 non-null  float32
 6   T24           67328 non-null  float32
 7   T30           67328 non-null  float32
 8   T48           67328 non-null  float32
 9   T50           67328 non-null  float32
 10  P15           67328 non-null  float32
 11  P2            67328 non-null  float32
 12  P21           67328 non-null  float32
 13  P24           67328 non-null  float32
 14  Ps30          67328 non-null  float32
 15  P40           67328 non-null  float32
 16  P50           67328 non-null  float32
 17  Nf            67328 non-null  float32
 18  Nc            67328 non-null  float32

In [17]:
df["hs"].value_counts()

hs
0    45670
1    21658
Name: count, dtype: int64

In [6]:
np.allclose(
    df["EPR"],
    df["P50"] / df["P2"]
)

True

### 2.1 Data Audit

In [7]:
print("Shape:", df.shape)
print("Engines:", df["unit"].nunique())
print("Flights:", df[["unit", "cycle"]].drop_duplicates().shape[0])

print("\nHealth status:")
print(df["hs"].value_counts())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(15))

Shape: (67328, 50)
Engines: 15
Flights: 994

Health status:
hs
0    45670
1    21658
Name: count, dtype: int64

Missing values:
split      0
raw_idx    0
alt        0
Mach       0
TRA        0
T2         0
T24        0
T30        0
T48        0
T50        0
P15        0
P2         0
P21        0
P24        0
Ps30       0
dtype: int64


### 2.2 Define Groups of Columns

In [8]:
metadata_cols = [
    "split",
    "raw_idx",
    "unit",
    "cycle"
]

evaluation_cols = [
    "hs",
    "Fc"
]

excluded_cols = [
    "RUL"   # remaining useful life; RUL prediction is out of scope
]

degradation_cols = [
    "fan_eff_mod", "fan_flow_mod",
    "LPC_eff_mod", "LPC_flow_mod",
    "HPC_eff_mod", "HPC_flow_mod",
    "HPT_eff_mod", "HPT_flow_mod",
    "LPT_eff_mod", "LPT_flow_mod"
]

operating_cols = [
    "alt",
    "Mach",
    "TRA",
    "T2"
]

sensor_cols = [
    "T24", "T30", "T48", "T50",
    "P15", "P2", "P21", "P24", "Ps30", "P40", "P50",
    "Nf", "Nc", "Wf"
]

engineered_cols = [
    "EPR"
]

## 3.0 Operating-Condition Variables

The dataset contains four variables describing operating conditions: `alt`, `Mach`, `TRA`, and `T2`.

These variables were retained for exploratory analysis and model diagnostics but were excluded from the initial Isolation Forest (IF) feature set. Because Isolation Forest (IF) identifies observations that are unusual in the joint feature space, uncommon but valid operating conditions could otherwise be interpreted as anomalous engine behavior.

Model performance will later be evaluated across flight classes and operating conditions to determine whether operating context materially affects anomaly scores.

In [9]:
flight_context = (
    df.groupby(["unit", "cycle"])
      .agg(
          hs=("hs", "first"),
          Fc=("Fc", "first"),
          alt_mean=("alt", "mean"),
          Mach_mean=("Mach", "mean"),
          TRA_mean=("TRA", "mean"),
          T2_mean=("T2", "mean")
      )
      .reset_index()
)

print("Flights:", len(flight_context))

print("\nHealth status by flight:")
print(flight_context["hs"].value_counts())

print("\nFlight class by health status:")
print(pd.crosstab(
    flight_context["Fc"],
    flight_context["hs"],
    margins=True
))

Flights: 994

Health status by flight:
hs
0    675
1    319
Name: count, dtype: int64

Flight class by health status:
hs     0    1  All
Fc                
1    249  143  392
2    219  101  320
3    207   75  282
All  675  319  994


### 3.1 Define Candidate Sensor Features

#### 3.1.0 Flight-Level Preparations

In [10]:
sensor_cols = [
    "T24", "T30", "T48", "T50",
    "P15", "P2", "P21", "P24", "Ps30", "P40", "P50",
    "Nf", "Nc", "Wf"
]

engineered_cols = [
    "EPR"
]

candidate_cols = sensor_cols + engineered_cols

df[candidate_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
T24,67328.0,572.130920,21.079077,484.381195,557.295044,570.268219,585.603210,633.277100
T30,67328.0,1335.746704,70.472115,1074.434448,1285.328369,1331.673218,1375.769989,1534.419678
T48,67328.0,1641.445435,128.309387,1240.318115,1548.016754,1650.446472,1720.214081,1990.432007
T50,67328.0,1136.916016,63.622227,892.898560,1092.000916,1128.430298,1171.488342,1357.166870
P15,67328.0,13.435461,2.805613,5.940456,11.003126,13.828550,15.535592,20.436853
P2,67328.0,10.542388,2.351471,4.402566,8.414203,11.049796,12.314898,14.733771
P21,67328.0,13.640062,2.848338,6.030920,11.170686,14.039137,15.772175,20.748074
P24,67328.0,16.500607,3.381960,6.937919,13.776450,16.708064,18.779948,26.346664
Ps30,67328.0,243.858185,59.919201,80.820160,199.252995,233.154762,278.265961,450.506134
P40,67328.0,248.128357,60.705673,82.565193,202.909176,237.440689,283.130508,456.574249


#### 3.1.1 Flight Sensor Means

In [11]:
flight_sensor_means = (
    df.groupby(["unit", "cycle"])[candidate_cols]
      .mean()
)

flight_sensor_means.shape

(994, 15)

#### 3.1.2 Calculate Correlations

In [12]:
corr = flight_sensor_means.corr()

### 3.2.0 Plotly Heatmap

In [13]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Correlation of Mean Sensor Measurements Across Flights"
)

fig.update_layout(
    width=900,
    height=800
)

fig.show()

### 3.2.1 High-Correlation Table

The heatmap above shows every pairwise correlation, but with 15 features it is hard to pick out which pairs are nearly redundant. The following table narrows the full correlation matrix down to only the **strongly correlated pairs**.

The code below:
1. Loops over each pair of features **once** (the matrix is symmetric, so the upper triangle is enough, and the diagonal of self-correlations is skipped).
2. Keeps only pairs with an absolute correlation of $|r| \geq 0.95$, meaning the two features move almost in lockstep across flights. The absolute value is used so that strong negative correlations would also be caught.
3. Sorts the remaining pairs from strongest to weakest.

A pair near $|r| = 1.0$ means one feature is almost an exact copy of the other at the flight-mean level. Keeping both adds little new information and can give that signal extra weight in distance- or split-based models. This table is therefore used to decide which sensors can be dropped with little loss of information (Section 3.2.2).

**Note:** These correlations are computed on raw flight means, where much of the shared variation comes from operating conditions (altitude, Mach, throttle) moving many sensors at once. Pairs that are nearly identical even so (e.g., `P15`-`P21`, `Ps30`-`P40`) are the safest to treat as redundant.

In [14]:
high_corr_pairs = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        r = corr.iloc[i, j]

        if abs(r) >= 0.95:
            high_corr_pairs.append({
                "feature_1": corr.columns[i],
                "feature_2": corr.columns[j],
                "correlation": r
            })

high_corr_df = (
    pd.DataFrame(high_corr_pairs)
      .sort_values("correlation", key=abs, ascending=False)
      .reset_index(drop=True)
)

high_corr_df

,feature_1,feature_2,correlation
0,P15,P21,1.000000
1,Ps30,P40,0.999956
2,P21,P24,0.998881
3,P15,P24,0.998881
4,P15,P2,0.998627
5,P2,P21,0.998627
6,P2,P50,0.996451
7,P15,P50,0.995435
8,P21,P50,0.995435
9,P2,P24,0.995319


### 3.2.2 Clusters of Correlations

At the $|r| \geq 0.95$ threshold, the candidate features fall into two high-correlation clusters. Both clusters make sense physically: the sensors in each one respond to the same underlying driver, so they rise and fall together from flight to flight.

**Cluster 1: Pressures, Fuel Flow, & LPC Exit Temperature**
`P2`, `P15`, `P21`, `P24`, `Ps30`, `P40`, `P50`, `T24`, `Wf`

| Feature | What It Measures |
|:-------:|------------------|
| `P2` | Total pressure at the fan inlet |
| `P15` | Total pressure in the bypass duct |
| `P21` | Total pressure at the fan exit |
| `P24` | Total pressure at the low-pressure compressor (LPC) exit |
| `Ps30` | Static pressure at the high-pressure compressor (HPC) exit |
| `P40` | Total pressure at the burner exit |
| `P50` | Total pressure at the low-pressure turbine (LPT) exit |
| `T24` | Total temperature at the LPC exit |
| `Wf` | Fuel flow |

These features all scale with **how much air is moving through the engine**. At lower altitudes the air is denser,
so the inlet pressure (`P2`) is higher, and every pressure further downstream rises with it, since each stage
compresses (or expands) the air it receives. Fuel flow follows the same pattern, because the engine burns fuel in
proportion to the airflow, and `T24` rises with the LPC pressure because compressing air heats it.

**Cluster 2: HPC Exit Temperature & Core Speed**
`T30`, `Nc`

| Feature | What it measures |
|---|---|
| `T30` | Total temperature at the HPC exit |
| `Nc` | Physical core (high-pressure shaft) speed |

These two are linked through the high-pressure compressor. When the core spins faster (`Nc`), the HPC compresses
the air more, and more compression means hotter air leaving the compressor (`T30`).

##### Features Outside Both Clusters

* `T48` (temperature between the high- and low-pressure turbines);
* `T50` (LPT exit temperature); and
* `Nf` (fan speed)

all did not exceed the threshold with any other feature, so each carries information not fully captured by the others.

##### Exclusions
The easiest exclusions to justify are the pairs that are effectively duplicates at the flight-mean level:
* `P21`, which is indistinguishable from `P15` ($r = 1.000$).
    * Fan exit pressure and bypass duct pressure are measured at nearly the same point in the airflow, so they carry the same information. `P15` is kept.
* `P40`, which is indistinguishable from `Ps30` ($r \approx 0.99996$).
    * The burner sits directly between the HPC exit and the burner exit, with little pressure change across it. `Ps30` is kept.

The remaining **Cluster 1** features are also highly correlated, but not identical, so they are kept for now. Their
small differences may still reflect differences in engine health, which is what the anomaly detection is looking for.

### 3.3 Final Candidate Sensor Features

Dropping `P21` and `P40` leaves the 12 physical sensors used as the starting feature set in IF Notebook #2.

In [15]:
candidate_sensor_cols = [
    "T24",
    "T30",
    "T48",
    "T50",
    "P15",
    "P2",
    "P24",
    "Ps30",
    "P50",
    "Nf",
    "Nc",
    "Wf"
]